In [65]:
# Importing libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import get_close_matches
from IPython.display import display, HTML

In [66]:
# Loading books.csv (Dataset that was obtained from https://github.com/zygmuntz/goodbooks-10k/)
books = pd.read_csv("books.csv")
books['title'] = books['title'].fillna('')
books['authors'] = books['authors'].fillna('')
books['image_url'] = books['image_url'].fillna('')
books.head()

,book_id,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780000e+12,Suzanne Collins,2008.0,The Hunger Games,...,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780000e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,...,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780000e+12,Stephenie Meyer,2005.0,Twilight,...,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780000e+12,Harper Lee,1960.0,To Kill a Mockingbird,...,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780000e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,...,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...


In [67]:
# Combine title and authors into a single content column
books['content'] = books['title'] + ' ' + books['authors']

In [68]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(books['content'])

In [69]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [70]:
def recommend_books_with_images(book_title, top_n=5):
    """Return top N similar books with images."""
    idx = books[books['title'].str.lower() == book_title.lower()].index
    if len(idx) == 0:
        return pd.DataFrame()
    idx = idx[0]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_indices = [i[0] for i in sim_scores[1:top_n+1]]
    
    recommended = books.iloc[top_indices][['title', 'authors', 'image_url']]
    return recommended

In [71]:
def find_book(title):
    """Return closest matching title from the dataset, asking user to choose."""
    titles = books['title'].tolist()
    matches = get_close_matches(title, titles, n=5, cutoff=0.6)
    if not matches:
        return None
    print("Did you mean one of these books?")
    for i, match in enumerate(matches):
        print(f"{i+1}. {match}")
    choice = input("Enter the number of the correct book, or 0 if none: ")
    try:
        choice = int(choice)
        if 1 <= choice <= len(matches):
            return matches[choice-1]
        else:
            return None
    except:
        return None

In [72]:
# User input for books
book_to_search = input("Enter a book title you like: ")
exact_title = find_book(book_to_search)

if exact_title:
    print(f"\nShowing recommendations for: '{exact_title}'\n")
    recommended_books = recommend_books_with_images(exact_title, top_n=5)
    
    # Display as HTML with images
    html_str = ""
    for _, row in recommended_books.iterrows():
        html_str += f"""
        <div style='display:inline-block; text-align:center; margin:10px;'>
            <img src="{row['image_url']}" width="120" height="180"><br>
            <b>{row['title']}</b><br>
            <i>{row['authors']}</i>
        </div>
        """
    display(HTML(html_str))
else:
    print("Book not found in the dataset.")

Enter a book title you like:  The Hunger Games


Did you mean one of these books?
1. The Hating Game
2. The Quiet Game
3. The Hungry Tide
4. The Hunger Games Tribute Guide
5. The Westing Game


Enter the number of the correct book, or 0 if none:  4



Showing recommendations for: 'The Hunger Games Tribute Guide'

